# เทรน YOLO-OBB หาเลขที่ดิน

1. เมนู **Runtime → Change runtime type → GPU**
2. อัปโหลด `land_yolo.zip` ไปที่ **My Drive** (โฟลเดอร์แรกสุด)
3. กด Run ทีละเซลล์จากบนลงล่าง

ใช้ข้อมูล **จำลองเท่านั้น** ห้ามอัปโหลดเอกสารจริงของผู้อื่น
YOLO ของ Ultralytics เป็น AGPL-3.0 และโมเดลตั้งต้นเทรนจาก DOTAv1 ซึ่งภาพต้นทางจำกัดการใช้เชิงวิชาการ

## 1. เช็ก GPU และเชื่อม Drive

In [ ]:
import torch
assert torch.cuda.is_available(), "ยังไม่ได้เลือก GPU: Runtime > Change runtime type"
print(torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, zipfile
ZIP = "/content/drive/MyDrive/land_yolo.zip"
OUT = "/content/drive/MyDrive/land_yolo_runs"   # โมเดลเก็บที่นี่ ไม่หายเมื่อเซสชันจบ
assert os.path.exists(ZIP), "ไม่พบ land_yolo.zip ใน My Drive"
os.makedirs(OUT, exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall("/content")
for s in ("train", "val"):
    print(s, len(os.listdir(f"/content/land_yolo/images/{s}")), "ภาพ")

## 2. ติดตั้ง

In [ ]:
%pip install -q ultralytics
import ultralytics
print("ultralytics", ultralytics.__version__)

## 3. ลองเทรน 1 รอบก่อน เพื่อดูว่าไม่มี error และใช้เวลากี่นาทีต่อรอบ

In [ ]:
import time
from ultralytics import YOLO
ARGS = dict(data="/content/land_yolo/data.yaml", imgsz=960, batch=8,
            fliplr=0.0, flipud=0.0,      # ห้ามกลับด้าน ตัวเลขจะผิดรูป
            degrees=5, scale=0.3, mosaic=0.3)
t = time.time()
YOLO("yolo26n-obb.pt").train(epochs=1, project=OUT, name="smoke", exist_ok=True, **ARGS)
print("นาทีต่อรอบ", round((time.time() - t) / 60, 1))

## 4. เทรนจริง

ปรับ `EPOCHS` จากเวลาที่วัดได้ หยุดเองเมื่อ val ไม่ดีขึ้น 10 รอบ
ถ้าเซสชันหลุด รันเซลล์ 1–2 ใหม่ แล้วข้ามไปเซลล์ "ทำต่อ"

In [ ]:
EPOCHS = 40
YOLO("yolo26n-obb.pt").train(epochs=EPOCHS, patience=10, save_period=5,
                            project=OUT, name="obb", exist_ok=True, **ARGS)

**ทำต่อ** (ใช้เฉพาะเมื่อเซสชันหลุดกลางทาง)

In [ ]:
# from ultralytics import YOLO
# YOLO(f"{OUT}/obb/weights/last.pt").train(resume=True)

## 5. คะแนนการหาบน val แยกคลาส

In [ ]:
best = YOLO(f"{OUT}/obb/weights/best.pt")
m = best.val(data="/content/land_yolo/data.yaml", imgsz=960, max_det=600)
for i, n in best.names.items():
    print(f"{n:8s}", "mAP50-95", round(float(m.box.maps[i]), 3))

## 6. ทำนายทั้งหน้า (val 120 แผ่น) แล้วดาวน์โหลดไปวัดการอ่านบน Mac

In [ ]:
%%writefile /content/yolo_tiles.py
"""Run a YOLO-OBB model over whole pages tile by tile (used inside the Colab notebook).

predict_page(detect, image) -> [{"cls": int, "conf": float, "quad": [[x, y] * 4]}]
`detect` takes a list of PIL tiles and returns, per tile, a list of (cls, conf, quad).
"""
import json
import math
from pathlib import Path

from PIL import Image

TILE, OVERLAP = 960, 160


def starts(size):
    if size <= TILE:
        return [0]
    s = list(range(0, size - TILE, TILE - OVERLAP))
    return s + [size - TILE]


def center(q):
    return sum(p[0] for p in q) / 4, sum(p[1] for p in q) / 4


def short_side(q):
    return min(math.dist(q[0], q[1]), math.dist(q[1], q[2]))


def predict_page(detect, im):
    boxes = [(x, y) for y in starts(im.height) for x in starts(im.width)]
    found = []
    for (x, y), dets in zip(boxes, detect([im.crop((x, y, x + TILE, y + TILE)) for x, y in boxes])):
        for cls, conf, q in dets:
            xs, ys = [p[0] for p in q], [p[1] for p in q]
            m = 3  # cut by an inner tile edge: the neighbouring tile has it whole
            if ((x > 0 and min(xs) < m) or (y > 0 and min(ys) < m)
                    or (x + TILE < im.width and max(xs) > TILE - m) or (y + TILE < im.height and max(ys) > TILE - m)):
                continue
            found.append(dict(cls=int(cls), conf=float(conf), quad=[[float(px + x), float(py + y)] for px, py in q]))
    keep = []
    for d in sorted(found, key=lambda d: -d["conf"]):
        c, s = center(d["quad"]), short_side(d["quad"])
        if all(math.dist(c, center(k["quad"])) > .5 * min(s, short_side(k["quad"])) for k in keep):
            keep.append(d)
    return keep


def ultralytics_detector(model, conf=.15, batch=6):
    def detect(tiles):
        out = []
        for i in range(0, len(tiles), batch):
            for r in model.predict(tiles[i:i + batch], imgsz=TILE, conf=conf, max_det=600, verbose=False):
                o = r.obb
                out.append([(int(c), float(p), q) for c, p, q in
                            zip(o.cls.int().tolist(), o.conf.tolist(), o.xyxyxyxy.tolist())])
        return out
    return detect


def run_folder(detect, folder, out):
    res = {}
    for p in sorted(Path(folder).rglob("*.jpg")):
        res[p.name] = predict_page(detect, Image.open(p).convert("RGB"))
    Path(out).write_text(json.dumps(res))
    print(len(res), "pages ->", out)

In [ ]:
import sys
sys.path.insert(0, "/content")
from yolo_tiles import run_folder, ultralytics_detector
run_folder(ultralytics_detector(best), "/content/land_yolo/val_pages", f"{OUT}/preds_val.json")
from google.colab import files
files.download(f"{OUT}/preds_val.json")
files.download(f"{OUT}/obb/weights/best.pt")

## 7. กลับมาที่ Mac

วาง `preds_val.json` ในโฟลเดอร์โปรเจกต์ แล้วรัน

```bash
python3 score_yolo.py --preds preds_val.json --split val --work /tmp/score
```